# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [ ]:
# !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [1]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

/Users/yasaman.nj/ironhack/labs/lab-agent-vector-store/.venv/langchain-v0.2/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [4]:
doc_path =  "./datasets/state_of_the_union.txt"

In [5]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [6]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [7]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [9]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [10]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import ChatOpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [12]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [13]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/var/folders/8z/s15bhnzx7zng08dcc75mh3_c0000gn/T/ipykernel_68310/1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [14]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.
Thought:I now know the final answer.  
Final Answer: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech, describing her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech, describing her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.'}

In [15]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and advantages of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why should someone use Ruff over Flake8 for Python linting?"
Observation: Someone might choose to use Ruff over Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Plugins Needed**: Ruff implements all rules natively and does not support custom or third-party rules, simplifying the setup process.

5. **Compatibility with Formatters**: Ruff is designed to work well alon

{'input': 'Why use ruff over flake8?',
 'output': "People might choose to use Ruff over Flake8 for Python linting due to its superior performance, comprehensive rule set (over 900 rules compared to Flake8's 409), automatic fixing capabilities, no need for custom plugins, compatibility with formatters like Black, potential as a drop-in replacement for Flake8, and support for modern Python versions (3.7 and above). Overall, Ruff provides a more extensive and efficient linting experience."}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [16]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [17]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [18]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the State of the Union address?"
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.'}

In [19]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about the advantages of using Ruff over Flake8, which are both Python linters. This will help me provide a comprehensive answer to the question. 
Action: Ruff QA System
Action Input: What are the advantages of using Ruff over Flake8?
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which allows for a broader range of linting capabilities.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Plugins Needed**: Ruff does not support custom lint rules, but it includes many rules natively that would require additional plugins in Flake8, simplifying the setup.

5. **Compatibility with Formatters**: Ruff is designed to work seamlessl

{'input': 'Why use ruff over flake8?',
 'output': 'The advantages of using Ruff over Flake8 include:\n\n1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which allows for a broader range of linting capabilities.\n\n3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.\n\n4. **No Custom Plugins Needed**: Ruff does not support custom lint rules, but it includes many rules natively that would require additional plugins in Flake8, simplifying the setup.\n\n5. **Compatibility with Formatters**: Ruff is designed to work seamlessly with formatters like Black, and it can defer stylistic rules that are handled by automated formatting.\n\n6. **Simplified Configuration**: Ruff uses different rule codes and prefixes to minimize conflicts across p

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [20]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [21]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [22]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about Ruff's functionality with Jupyter Notebooks.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:I have found that Ruff integrates with nbQA to run over Jupyter Notebooks. Now, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address, but the previous action did not yield a clear answer. I will try to rephrase the question to see if I can get more information.

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, and the president did not mention that tool in the State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.

